In [ ]:
# Use this script after updating the claims number all time data in a local gdb but before uploading it to AGOL.
# You will use this script to take the claims number all time data and pivot it by industry. Once you're done with
# this script (and any others that need to be run for this data), you can then upload both the tall and the pivoted
# data at once.

# Background:
# The claims number all time data is formatted such that each different combination of geography, injury type, age group,
# etc. are different records. This is fine for a table (though it makes for a very long table), but it's way too much data
# to display as a joined hosted feature layer. For this reason, on the data portal, pivoted data is used for the map.

# Ideas for improvement:
# Bypass Excel. I'm only using it because this works well enough and I have other stuff to do.

In [ ]:
# Used for re-shaping the table
import pandas as pd 

# Used for working with Excel
import openpyxl 

In [ ]:
# You will need to have an ArcGIS Pro project open
aprx = arcpy.mp.ArcGISProject("CURRENT")

# In that project, you will need to have a map called "Map" (or change this name)
m = aprx.listMaps("Map")[0]

# In that map, you will need to have a table called "Workers_Compensation_Claims_NORA_Number_All_Time" (or change this name)
claimsData = m.listTables("Workers_Compensation_Claims_NORA_Number_All_Time")[0]

In [ ]:
# Geodatabase location & where you want to save any Excel files
location = r"C:\Users\JohnsonN35\Local_work\Copied_from_local_work\MiTracking_Exploration"

# File name
file = "Claims_all_time_pivoted_by_industry_number"

# Excel extension
ext = ".xlsx"

# Geodatabase name (this script assumes your gdb is in the same place as where you want to save any Excel files)
gdb = "MiTracking_Workers_Compensation_Claims_NORA_Number_All_Time.gdb"

In [ ]:
# Create a definition query for only the all-injuries, all-ages, etc. records

claimsDataDQ = '"Injury_Type" = \'01\' And "Age_Group" = \'1\' And "Sex" = \'1\' And "Employer_Size" = \'01\''

claimsData.definitionQuery = claimsDataDQ

In [ ]:
# Create a pandas dataframe

fields = ['FIPS_Table', 'Time_Period', 'Industry_NORA', 'Number_Claims'] # Leaving out injury type, age group, sex, and employer size

df = pd.DataFrame.from_records(data = arcpy.da.SearchCursor(claimsData, fields), columns = fields)

df.head()

In [ ]:
# Create a wide dataframe

df_wide = df.pivot(index = ['FIPS_Table', 'Time_Period'], columns = ['Industry_NORA'], values = ['Number_Claims'])

df_wide.reset_index(inplace = True)

df_wide.head()

In [ ]:
# Export to an Excel spreadsheet

name = file + "_pandas"

df_wide.to_excel(location + "/" + name + ext, sheet_name = 'Sheet1')

In [ ]:
# Clean up the Excel file (we can't use the current table structure in a feature class)

from openpyxl import load_workbook

outpath = location + "/" + name + ext
wb = load_workbook(outpath)
ws = wb.active

# Start all the NORA sector fields with "NORA"
# (i.e., if estimates go from cell D2 (4th column) - N2 (14th column), put 14 next to the break)
for index, row in enumerate(ws.rows, start = 4):
        if index > 14: break
        ws.cell(row = 2, column = index).value = 'NORA_' + ws.cell(row = 2, column = index).value

# Write the values of cells B1 (2nd column) - C1 (3rd column) into cells B2 - C2
for index, row in enumerate(ws.rows, start = 2):
        if index > 3: break
        ws.cell(row = 2, column = index).value = ws.cell(row = 1, column = index).value

# Unmerge estimate cells (causes issues with deleting rows in next step)
ws.unmerge_cells('D1:N1')

# Delete rows and columns we don't need and that can't be imported into a feature class
ws.delete_cols(1)
ws.delete_rows(1)
ws.delete_rows(2) # Once we delete the first row, the original third row becomes the second row
wb.save(outpath)

In [ ]:
# Create a new table to hold the wide sheet

tableName = "Claims_all_time_pivoted_by_NORA_number"

arcpy.management.CreateTable(location + "/" + gdb, tableName)

In [ ]:
# Create the fields for the table
arcpy.management.AddFields(tableName, "\
FIPS_Table TEXT 'FIPS' 7 # #;\
Geographic_Area TEXT 'Geographic Area' 50 # #;\
Time_Period TEXT 'Time Period' 9 # #;\
NORA_01 LONG 'NORA 01' # # #;\
NORA_02 LONG 'NORA 02' # # #;\
NORA_03 LONG 'NORA 03' # # #;\
NORA_04 LONG 'NORA 04' # # #;\
NORA_05 LONG 'NORA 05' # # #;\
NORA_06 LONG 'NORA 06' # # #;\
NORA_07 LONG 'NORA 07' # # #;\
NORA_08 LONG 'NORA 08' # # #;\
NORA_09 LONG 'NORA 09' # # #;\
NORA_10 LONG 'NORA 10' # # #;\
NORA_11 LONG 'NORA 11' # # #;\
")

In [ ]:
# Append wide sheet to new table

spreadsheet = location + "\\" + name + ext + "\\Sheet1$"

fieldMapping = '\
    FIPS_Table "FIPS" true true false 7 Text 0 0,First,#,spreadsheet,FIPS_Table,0,254;\
    Geographic_Area "Geographic Area" true true false 50 Text 0 0,First,#,spreadsheet,Geographic_Area,0,254;\
    Time_Period "Time Period" true true false 9 Text 0 0,First,#,spreadsheet,Time_Period,0,254;\
    NORA_01 "NORA 01" true true false 4 Long 0 0,First,#,spreadsheet,NORA_01,-1,-1;\
    NORA_02 "NORA 02" true true false 4 Long 0 0,First,#,spreadsheet,NORA_02,-1,-1;\
    NORA_03 "NORA 03" true true false 4 Long 0 0,First,#,spreadsheet,NORA_03,-1,-1;\
    NORA_04 "NORA 04" true true false 4 Long 0 0,First,#,spreadsheet,NORA_04,-1,-1;\
    NORA_05 "NORA 05" true true false 4 Long 0 0,First,#,spreadsheet,NORA_05,-1,-1;\
    NORA_06 "NORA 06" true true false 4 Long 0 0,First,#,spreadsheet,NORA_06,-1,-1;\
    NORA_07 "NORA 07" true true false 4 Long 0 0,First,#,spreadsheet,NORA_07,-1,-1;\
    NORA_08 "NORA 08" true true false 4 Long 0 0,First,#,spreadsheet,NORA_08,-1,-1;\
    NORA_09 "NORA 09" true true false 4 Long 0 0,First,#,spreadsheet,NORA_09,-1,-1;\
    NORA_10 "NORA 10" true true false 4 Long 0 0,First,#,spreadsheet,NORA_10,-1,-1;\
    NORA_11 "NORA 11" true true false 4 Long 0 0,First,#,spreadsheet,NORA_11,-1,-1\
    '

arcpy.management.Append(spreadsheet, tableName, "NO_TEST", fieldMapping)

In [ ]:
# Create local copy of the appropriate geography layer, join by FIPS, and field-calculate geographic area names into pivoted layer:
# https://mdhhs.maps.arcgis.com/home/item.html?id=ec983f14c3394a53a67711b9ffa219f2